# Tech Tree Usability Testing — Data Cleaning
This notebook reads the raw Google Form CSV, cleans it, and exports a tidy CSV for the Streamlit dashboard.

In [ ]:
import pandas as pd

df = pd.read_csv('../Tech Tree Usability Testing (Responses) - Form Responses 1.csv')
df.head()

In [ ]:
# Rename columns to short, code-friendly names
df.columns = [
    'timestamp',
    'name',
    'age_group',
    'background',
    'experience_rating',
    'understood_homepage',
    'explore_without_instructions',
    'difficulty_node_types',
    'wanted_back_reset',
    'visual_cues_helpful',
    'would_recommend',
    'describe_experience',
    'additional_feedback'
]
df.head()

In [ ]:
# Parse timestamp
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%m/%d/%Y %H:%M:%S')

# Ensure experience_rating is numeric
df['experience_rating'] = pd.to_numeric(df['experience_rating'], errors='coerce')

df.dtypes

In [ ]:
def normalize_yes_no(val):
    """Map free-text answers to Yes / No / Unclear."""
    if pd.isna(val):
        return 'Unclear'
    v = str(val).strip().lower()
    if v in ('yes', 'immediately', 'kinda', 'pretty clear.', 'yes.', 'yes '):
        return 'Yes'
    if v in ('no', 'no.', 'not really', 'na', 'n/a'):
        return 'No'
    # Longer answers that start with yes/no
    if v.startswith('yes'):
        return 'Yes'
    if v.startswith('no'):
        return 'No'
    return 'Unclear'

yes_no_cols = [
    'understood_homepage',
    'explore_without_instructions',
    'difficulty_node_types',
    'wanted_back_reset',
    'visual_cues_helpful',
]

for col in yes_no_cols:
    df[col] = df[col].apply(normalize_yes_no)

# would_recommend has Yes / No / Maybe
def normalize_recommend(val):
    if pd.isna(val):
        return 'Unclear'
    v = str(val).strip().lower()
    if v.startswith('yes'):
        return 'Yes'
    if v.startswith('no'):
        return 'No'
    if v.startswith('maybe'):
        return 'Maybe'
    return 'Unclear'

df['would_recommend'] = df['would_recommend'].apply(normalize_recommend)

df[yes_no_cols + ['would_recommend']]

In [ ]:
# Fill NaN text fields with empty string
text_cols = ['background', 'describe_experience', 'additional_feedback']
for col in text_cols:
    df[col] = df[col].fillna('')

df.info()

In [ ]:
# Export cleaned CSV
df.to_csv('cleaned_responses.csv', index=False)
print('Saved cleaned_responses.csv ✓')
df